In [ ]:
# ============================================================
# EXPERIMENT 12A: ADABOOST CLASSIFICATION FROM SCRATCH
# Binary Classification - Weak Learners: Decision Stumps
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

# Step 1 & 2: Create Synthetic Dataset and Train-Test Split
X = np.array([[1.0, 55], [1.5, 58], [2.0, 60], [2.5, 63], [3.0, 65], [3.5, 67],
              [4.0, 70], [4.5, 72], [5.0, 75], [5.5, 78], [6.0, 80], [6.5, 82],
              [7.0, 85], [7.5, 88], [8.0, 90], [8.5, 92]], dtype=float)
y = np.array([-1] * 8 + [1] * 8)

np.random.seed(42)
idx = np.random.permutation(len(X))
split = int(0.80 * len(X))
X_train, y_train = X[idx[:split]], y[idx[:split]]
X_test, y_test = X[idx[split:]], y[idx[split:]]
print(f"Training Samples: {len(X_train)}\nTesting Samples: {len(X_test)}")

# Step 3: Decision Stump (Weak Learner)
class DecisionStump:
    def __init__(self, feature_index=None, threshold=None, polarity=1):
        self.feature_index, self.threshold, self.polarity = feature_index, threshold, polarity

    def predict(self, X):
        col = X[:, self.feature_index]
        return np.where(col < self.threshold, -1.0, 1.0) if self.polarity == 1 else np.where(col >= self.threshold, -1.0, 1.0)

# Step 4: AdaBoost Classifier
class AdaBoost:
    def __init__(self, n_estimators=10):
        self.n_estimators = n_estimators
        self.stumps, self.alphas, self.errors = [], [], []

    def fit(self, X, y):
        n_samples, n_features = X.shape
        weights = np.ones(n_samples) / n_samples
        for it in range(self.n_estimators):
            stump, best_error = DecisionStump(), float("inf")
            for f in range(n_features):
                for t in np.unique(X[:, f]):
                    for p in (1, -1):
                        cand = DecisionStump(f, t, p)
                        error = np.sum(weights[cand.predict(X) != y])
                        if error < best_error:
                            best_error, stump = error, cand
            eps = 1e-10
            alpha = 0.5 * np.log((1 - best_error + eps) / (best_error + eps))
            weights *= np.exp(-alpha * y * stump.predict(X))
            weights /= np.sum(weights)
            self.stumps.append(stump), self.alphas.append(alpha), self.errors.append(best_error)
            print(f"Iteration {it+1}: Error={best_error:.4f}, Alpha={alpha:.4f}")

    def predict(self, X):
        return np.sign(sum(a * s.predict(X) for s, a in zip(self.stumps, self.alphas)))

# Step 5-7: Create Model, Train, and Make Predictions
model = AdaBoost(n_estimators=10)
print("\n===================================\nTRAINING ADABOOST\n===================================")
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# Step 8 & 9: Display Predictions and Accuracy
print("\nActual vs Predicted\n==============================\nActual\tPredicted")
for actual, predicted in zip(y_test, y_pred):
    print(f"{int(actual)}\t{int(predicted)}")

accuracy = np.mean(y_pred == y_test)
print("\n===================================\nADABOOST PERFORMANCE\n===================================")
print(f"Correct Predictions: {np.sum(y_pred == y_test)}\nTotal Predictions: {len(y_test)}\nAccuracy: {round(accuracy, 4)}\nAccuracy (%): {round(accuracy * 100, 2)}")

# Step 10-13: Confusion Matrix and Classification Metrics
TP = np.sum((y_test == 1) & (y_pred == 1))
TN = np.sum((y_test == -1) & (y_pred == -1))
FP = np.sum((y_test == -1) & (y_pred == 1))
FN = np.sum((y_test == 1) & (y_pred == -1))

print(f"\nConfusion Matrix\n-------------------------\nPredicted\n         -1  +1\nActual -1 {TN:3d} {FP:3d}\nActual +1 {FN:3d} {TP:3d}")

prec = TP / (TP + FP) if (TP + FP) else 0
rec = TP / (TP + FN) if (TP + FN) else 0
f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0
print(f"\nClassification Metrics\nPrecision: {round(prec, 4)}\nRecall: {round(rec, 4)}\nF1 Score: {round(f1, 4)}")

# Step 14: Display Weak Learner Information
print("\n===================================\nWEAK LEARNERS\n===================================")
for i, (stump, alpha, error) in enumerate(zip(model.stumps, model.alphas, model.errors)):
    print(f"\nStump {i+1}\nFeature: {stump.feature_index}\nThreshold: {stump.threshold}\nPolarity: {stump.polarity}\nError: {round(error, 4)}\nAlpha: {round(alpha, 4)}")

# Step 15: Predict New Student
new_student = np.array([[5.0, 76]])
new_pred = model.predict(new_student)[0]
print("\n===================================\nNEW STUDENT PREDICTION\n===================================")
print(f"Study Hours: {new_student[0, 0]}\nAttendance: {new_student[0, 1]}\nPredicted Class: {int(new_pred)}\nResult: {'PASS' if new_pred == 1 else 'FAIL'}")

# Step 16: Plot Training Error
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(model.errors) + 1), model.errors, marker="o")
plt.xlabel("Boosting Iteration"), plt.ylabel("Weighted Error"), plt.title("AdaBoost: Weak Learner Error")
plt.grid(True), plt.tight_layout(), plt.show()

# Step 17: Plot Alpha Values
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(model.alphas) + 1), model.alphas, marker="o")
plt.xlabel("Boosting Iteration"), plt.ylabel("Alpha"), plt.title("AdaBoost: Weak Learner Weights")
plt.grid(True), plt.tight_layout(), plt.show()

# Step 18: Plot Dataset
plt.figure(figsize=(8, 6))
plt.scatter(X[y == -1, 0], X[y == -1, 1], label="Fail", marker="o")
plt.scatter(X[y == 1, 0], X[y == 1, 1], label="Pass", marker="x")
plt.scatter(new_student[0, 0], new_student[0, 1], marker="*", s=180, label="New Student")
plt.xlabel("Study Hours"), plt.ylabel("Attendance (%)"), plt.title("AdaBoost Classification")
plt.legend(), plt.grid(True), plt.tight_layout(), plt.show()

In [ ]:
# ============================================================
# EXPERIMENT 12B: ADABOOST CLASSIFICATION USING SCIKIT-LEARN
# Real-World Dataset: Breast Cancer Wisconsin Dataset
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Step 1-3: Load Dataset, Create DataFrame, and Display Info
data = load_breast_cancer()
X, y = data.data, data.target
feature_names, target_names = data.feature_names, data.target_names
class_map = dict(enumerate(target_names))

df = pd.DataFrame(X, columns=feature_names)
df["Target"] = y
df["Class"] = df["Target"].map(class_map)

print(f"Dataset Shape:\n{X.shape}\n\nTarget Classes:\n{target_names}")
print(f"\nFirst Five Records:\n{df.head()}")
print(f"\nClass Distribution:\n{df['Class'].value_counts()}\n\nDataset Description:\n{df.describe()}")

# Step 4-5: Train-Test Split (Stratified)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print(f"\nTraining Samples: {X_train.shape[0]}\nTesting Samples: {X_test.shape[0]}")

# Step 6-8: Create Weak Learner (Decision Stump), AdaBoost Model, and Train
base_model = DecisionTreeClassifier(max_depth=1, random_state=42)
model = AdaBoostClassifier(estimator=base_model, n_estimators=50, learning_rate=1.0, random_state=42).fit(X_train, y_train)

# Step 9-11: Predictions, Probabilities, and Actual vs Predicted
y_pred, y_prob = model.predict(X_test), model.predict_proba(X_test)
results = pd.DataFrame({"Actual": y_test, "Predicted": y_pred, "Probability_Class_0": y_prob[:, 0], "Probability_Class_1": y_prob[:, 1]})
results["Actual Class"], results["Predicted Class"] = results["Actual"].map(class_map), results["Predicted"].map(class_map)
print(f"\nActual vs Predicted:\n{results.head(10)}")

# Step 12: Calculate and Display Accuracy
acc = accuracy_score(y_test, y_pred)
print("\n===================================\nADABOOST PERFORMANCE\n===================================")
print(f"Accuracy: {round(acc, 4)}\nAccuracy (%): {round(acc * 100, 2)}")

# Steps 13-14: Confusion Matrix and Classification Report
cm = confusion_matrix(y_test, y_pred)
print(f"\nConfusion Matrix:\n{cm}")
print(f"\nClassification Report:\n{classification_report(y_test, y_pred, target_names=target_names)}")

# Steps 15-17: Weak Learner Count, Weights, and Errors
print(f"\nNumber of Weak Learners:\n{len(model.estimators_)}\n\nWeak Learner Weights:\n{model.estimator_weights_}\n\nWeak Learner Errors:\n{model.estimator_errors_}")

# Steps 18-19: Feature Importance and Top 10 Features
importance = pd.DataFrame({"Feature": feature_names, "Importance": model.feature_importances_}).sort_values("Importance", ascending=False)
top_features = importance.head(10)
print(f"\nFeature Importance:\n{importance}\n\nTop 10 Important Features:\n{top_features}")

# Step 20: Feature Importance Plot
plt.figure(figsize=(9, 6))
plt.barh(top_features["Feature"][::-1], top_features["Importance"][::-1])
plt.xlabel("Importance"), plt.ylabel("Feature"), plt.title("AdaBoost Feature Importance")
plt.tight_layout(), plt.show()

# Step 21: Confusion Matrix Visualization
plt.figure(figsize=(7, 5))
plt.imshow(cm, interpolation="nearest"), plt.title("AdaBoost Confusion Matrix")
plt.xlabel("Predicted Class"), plt.ylabel("Actual Class")
plt.xticks([0, 1], target_names), plt.yticks([0, 1], target_names)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center")
plt.colorbar(), plt.tight_layout(), plt.show()

# Step 22: Actual vs Predicted Plot
idx = np.arange(len(y_test))
plt.figure(figsize=(9, 5))
plt.scatter(idx, y_test, label="Actual", marker="o")
plt.scatter(idx, y_pred, label="Predicted", marker="x")
plt.xlabel("Test Sample Number"), plt.ylabel("Class"), plt.title("AdaBoost: Actual vs Predicted")
plt.yticks([0, 1], target_names), plt.legend(), plt.grid(True), plt.tight_layout(), plt.show()

# Step 23: Accuracy During Boosting Iterations
staged_accuracy = [accuracy_score(y_test, p) for p in model.staged_predict(X_test)]
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(staged_accuracy) + 1), staged_accuracy, marker="o")
plt.xlabel("Boosting Iteration"), plt.ylabel("Test Accuracy"), plt.title("AdaBoost: Accuracy vs Boosting Iteration")
plt.grid(True), plt.tight_layout(), plt.show()

# Step 24: Predict a New Observation
new_sample = X_test[0].reshape(1, -1)
new_pred, new_prob = model.predict(new_sample)[0], model.predict_proba(new_sample)[0]
print("\n===================================\nNEW SAMPLE PREDICTION\n===================================")
print(f"Predicted Class: {target_names[new_pred]}\nProbability of Class 0: {round(new_prob[0], 4)}\nProbability of Class 1: {round(new_prob[1], 4)}")

In [ ]:
# ============================================================
# EXPERIMENT 12C: GRADIENT BOOSTING CLASSIFICATION USING SCIKIT-LEARN
# Real-World Dataset: Breast Cancer Wisconsin Dataset
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Step 1-3: Load Dataset, Create DataFrame, and Display Info
data = load_breast_cancer()
X, y = data.data, data.target
feature_names, target_names = data.feature_names, data.target_names
class_map = dict(enumerate(target_names))

df = pd.DataFrame(X, columns=feature_names)
df["Target"] = y
df["Class"] = df["Target"].map(class_map)

print(f"Dataset Shape:\n{X.shape}\n\nTarget Classes:\n{target_names}")
print(f"\nFirst Five Records:\n{df.head()}")
print(f"\nClass Distribution:\n{df['Class'].value_counts()}\n\nDataset Description:\n{df.describe()}")

# Step 4-5: Train-Test Split (Stratified)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print(f"\nTraining Samples: {X_train.shape[0]}\nTesting Samples: {X_test.shape[0]}")

# Step 6-7: Create and Train Gradient Boosting Model
model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42).fit(X_train, y_train)

# Step 8-10: Predictions, Probabilities, and Actual vs Predicted
y_pred, y_prob = model.predict(X_test), model.predict_proba(X_test)
results = pd.DataFrame({"Actual": y_test, "Predicted": y_pred, "Probability_Class_0": y_prob[:, 0], "Probability_Class_1": y_prob[:, 1]})
results["Actual Class"], results["Predicted Class"] = results["Actual"].map(class_map), results["Predicted"].map(class_map)
print(f"\nActual vs Predicted:\n{results.head(10)}")

# Step 11: Calculate and Display Accuracy
acc = accuracy_score(y_test, y_pred)
print("\n===================================\nGRADIENT BOOSTING PERFORMANCE\n===================================")
print(f"Accuracy: {round(acc, 4)}\nAccuracy (%): {round(acc * 100, 2)}")

# Steps 12-13: Confusion Matrix and Classification Report
cm = confusion_matrix(y_test, y_pred)
print(f"\nConfusion Matrix:\n{cm}")
print(f"\nClassification Report:\n{classification_report(y_test, y_pred, target_names=target_names)}")

# Steps 14-15: Feature Importance and Top 10 Features
importance = pd.DataFrame({"Feature": feature_names, "Importance": model.feature_importances_}).sort_values("Importance", ascending=False)
top_features = importance.head(10)
print(f"\nFeature Importance:\n{importance}\n\nTop 10 Important Features:\n{top_features}")

# Step 16: Feature Importance Plot
plt.figure(figsize=(9, 6))
plt.barh(top_features["Feature"][::-1], top_features["Importance"][::-1])
plt.xlabel("Importance"), plt.ylabel("Feature"), plt.title("Gradient Boosting Feature Importance")
plt.tight_layout(), plt.show()

# Step 17: Confusion Matrix Visualization
plt.figure(figsize=(7, 5))
plt.imshow(cm, interpolation="nearest"), plt.title("Gradient Boosting Confusion Matrix")
plt.xlabel("Predicted Class"), plt.ylabel("Actual Class")
plt.xticks([0, 1], target_names), plt.yticks([0, 1], target_names)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center")
plt.colorbar(), plt.tight_layout(), plt.show()

# Step 18: Actual vs Predicted Plot
idx = np.arange(len(y_test))
plt.figure(figsize=(9, 5))
plt.scatter(idx, y_test, label="Actual", marker="o")
plt.scatter(idx, y_pred, label="Predicted", marker="x")
plt.xlabel("Test Sample Number"), plt.ylabel("Class"), plt.title("Gradient Boosting: Actual vs Predicted")
plt.yticks([0, 1], target_names), plt.legend(), plt.grid(True), plt.tight_layout(), plt.show()

# Step 19: Accuracy During Boosting Iterations
staged_accuracy = [accuracy_score(y_test, p) for p in model.staged_predict(X_test)]
plt.figure(figsize=(8, 5))
plt.plot(range(1, len(staged_accuracy) + 1), staged_accuracy, marker="o")
plt.xlabel("Boosting Iteration"), plt.ylabel("Test Accuracy"), plt.title("Gradient Boosting: Accuracy vs Iteration")
plt.grid(True), plt.tight_layout(), plt.show()

# Step 20: Predict a New Observation
new_sample = X_test[0].reshape(1, -1)
new_pred, new_prob = model.predict(new_sample)[0], model.predict_proba(new_sample)[0]
print("\n===================================\nNEW SAMPLE PREDICTION\n===================================")
print(f"Predicted Class: {target_names[new_pred]}\nProbability of Class 0: {round(new_prob[0], 4)}\nProbability of Class 1: {round(new_prob[1], 4)}")

In [ ]:
# ============================================================
# EXPERIMENT 12D: XGBOOST CLASSIFICATION
# Real-World Dataset: Breast Cancer Wisconsin Dataset
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Step 1-3: Load Dataset, Create DataFrame, and Display Info
data = load_breast_cancer()
X, y = data.data, data.target
feature_names, target_names = data.feature_names, data.target_names
class_map = dict(enumerate(target_names))

df = pd.DataFrame(X, columns=feature_names)
df["Target"] = y
df["Class"] = df["Target"].map(class_map)

print(f"Dataset Shape:\n{X.shape}\n\nTarget Classes:\n{target_names}")
print(f"\nFirst Five Records:\n{df.head()}")
print(f"\nClass Distribution:\n{df['Class'].value_counts()}\n\nDataset Description:\n{df.describe()}")

# Step 4-5: Train-Test Split (Stratified)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print(f"\nTraining Samples: {X_train.shape[0]}\nTesting Samples: {X_test.shape[0]}")

# Step 6-7: Create and Train XGBoost Model
model = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, subsample=0.8,
                      colsample_bytree=0.8, objective="binary:logistic", eval_metric="logloss",
                      random_state=42).fit(X_train, y_train)

# Step 8-10: Predictions, Probabilities, and Actual vs Predicted
y_pred, y_prob = model.predict(X_test), model.predict_proba(X_test)
results = pd.DataFrame({"Actual": y_test, "Predicted": y_pred, "Probability_Class_0": y_prob[:, 0], "Probability_Class_1": y_prob[:, 1]})
results["Actual Class"], results["Predicted Class"] = results["Actual"].map(class_map), results["Predicted"].map(class_map)
print(f"\nActual vs Predicted:\n{results.head(10)}")

# Step 11: Calculate and Display Accuracy
acc = accuracy_score(y_test, y_pred)
print("\n===================================\nXGBOOST CLASSIFICATION PERFORMANCE\n===================================")
print(f"Accuracy: {round(acc, 4)}\nAccuracy (%): {round(acc * 100, 2)}")

# Steps 12-13: Confusion Matrix and Classification Report
cm = confusion_matrix(y_test, y_pred)
print(f"\nConfusion Matrix:\n{cm}")
print(f"\nClassification Report:\n{classification_report(y_test, y_pred, target_names=target_names)}")

# Steps 14-15: Feature Importance and Top 10 Features
importance = pd.DataFrame({"Feature": feature_names, "Importance": model.feature_importances_}).sort_values("Importance", ascending=False)
top_features = importance.head(10)
print(f"\nFeature Importance:\n{importance}\n\nTop 10 Important Features:\n{top_features}")

# Step 16: Feature Importance Plot
plt.figure(figsize=(9, 6))
plt.barh(top_features["Feature"][::-1], top_features["Importance"][::-1])
plt.xlabel("Importance"), plt.ylabel("Feature"), plt.title("XGBoost Feature Importance")
plt.tight_layout(), plt.show()

# Step 17: Confusion Matrix Visualization
plt.figure(figsize=(7, 5))
plt.imshow(cm, interpolation="nearest"), plt.title("XGBoost Confusion Matrix")
plt.xlabel("Predicted Class"), plt.ylabel("Actual Class")
plt.xticks([0, 1], target_names), plt.yticks([0, 1], target_names)
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center")
plt.colorbar(), plt.tight_layout(), plt.show()

# Step 18: Actual vs Predicted Plot
idx = np.arange(len(y_test))
plt.figure(figsize=(9, 5))
plt.scatter(idx, y_test, label="Actual", marker="o")
plt.scatter(idx, y_pred, label="Predicted", marker="x")
plt.xlabel("Test Sample Number"), plt.ylabel("Class"), plt.title("XGBoost: Actual vs Predicted")
plt.yticks([0, 1], target_names), plt.legend(), plt.grid(True), plt.tight_layout(), plt.show()

# Step 19: Predict a New Observation
new_sample = X_test[0].reshape(1, -1)
new_pred, new_prob = model.predict(new_sample)[0], model.predict_proba(new_sample)[0]
print("\n===================================\nNEW SAMPLE PREDICTION\n===================================")
print(f"Predicted Class: {target_names[new_pred]}\nProbability of Class 0: {round(new_prob[0], 4)}\nProbability of Class 1: {round(new_prob[1], 4)}")